In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DockerPySparkTest")
    .master("local[*]")
    .getOrCreate()
)

data = [
    ("Alice", 30),
    ("Bob", 25),
    ("Charlie", 35),
]

df = spark.createDataFrame(data, ["name", "age"])

df.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 30|
|    Bob| 25|
|Charlie| 35|
+-------+---+



In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BigDataLab1").getOrCreate()

movies_df = spark.read.csv("movies.csv", header=True, inferSchema=True)
ratings_df = spark.read.csv("ratings.csv", header=True, inferSchema=True)


In [6]:
movies_df.head(5)

[Row(movieId=1, title='The Dark Knight', genre='Acțiune'),
 Row(movieId=2, title='Inception', genre='Comedie'),
 Row(movieId=3, title='The Matrix', genre='Dramă'),
 Row(movieId=4, title='Gladiator', genre='Animație'),
 Row(movieId=5, title='Die Hard', genre='Acțiune')]

In [7]:
# Selectarea coloanelor
movies_df.select("title", "genre").show(5)

# Filtrarea rândurilor
animation_movies = movies_df.filter(movies_df.genre == "Animație")
animation_movies.show(5)

# Filtrarea rating-urilor >= 4.5
high_ratings = ratings_df.filter(ratings_df.rating >= 4.5)
high_ratings.show(5)


+---------------+--------+
|          title|   genre|
+---------------+--------+
|The Dark Knight| Acțiune|
|      Inception| Comedie|
|     The Matrix|   Dramă|
|      Gladiator|Animație|
|       Die Hard| Acțiune|
+---------------+--------+
only showing top 5 rows
+-------+------------+--------+
|movieId|       title|   genre|
+-------+------------+--------+
|      4|   Gladiator|Animație|
|      8|   Airplane!|Animație|
|     12|Forrest Gump|Animație|
|     16|   Toy Story|Animație|
|     20|      WALL-E|Animație|
+-------+------------+--------+

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     7|      6|   4.9|
|     2|      3|   4.7|
|     6|      1|   4.7|
+------+-------+------+



In [8]:
from pyspark.sql.functions import avg, count

# Rating mediu per film
avg_ratings = ratings_df.groupBy("movieId").agg(avg("rating").alias("avg_rating"))
avg_ratings.show(5)

# Număr rating-uri per film
count_ratings = ratings_df.groupBy("movieId").count()
count_ratings.show(5)

+-------+------------------+
|movieId|        avg_rating|
+-------+------------------+
|     12|               3.3|
|      1|              2.85|
|     16|2.8666666666666667|
|      6|3.7249999999999996|
|      3|3.0799999999999996|
+-------+------------------+
only showing top 5 rows
+-------+-----+
|movieId|count|
+-------+-----+
|     12|    2|
|      1|    2|
|     16|    3|
|      6|    4|
|      3|    5|
+-------+-----+
only showing top 5 rows


In [9]:
# Join filme și rating-uri
movie_ratings = movies_df.join(ratings_df, on="movieId", how="inner")
movie_ratings.show(5)

# Rating mediu per gen
genre_avg = movie_ratings.groupBy("genre").agg(avg("rating").alias("avg_genre_rating"))
genre_avg.show()

+-------+---------------+-------+------+------+
|movieId|          title|  genre|userId|rating|
+-------+---------------+-------+------+------+
|      1|The Dark Knight|Acțiune|     6|   4.7|
|      1|The Dark Knight|Acțiune|    10|   1.0|
|      3|     The Matrix|  Dramă|     1|   2.5|
|      3|     The Matrix|  Dramă|     2|   4.7|
|      3|     The Matrix|  Dramă|     4|   2.7|
+-------+---------------+-------+------+------+
only showing top 5 rows
+--------+------------------+
|   genre|  avg_genre_rating|
+--------+------------------+
| Comedie| 3.023076923076923|
|Animație| 2.330769230769231|
| Acțiune|2.8285714285714287|
|   Dramă| 2.564705882352941|
+--------+------------------+



In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSQLDemo").getOrCreate()
movies_df = spark.read.csv("movies.csv", header=True, inferSchema=True)
ratings_df = spark.read.csv("ratings.csv", header=True, inferSchema=True)
movies_df.createOrReplaceTempView("movies")
ratings_df.createOrReplaceTempView("ratings")
# Toate filmele de gen "Acțiune"
action_movies = spark.sql("SELECT * FROM movies WHERE genre = 'Acțiune'")
action_movies.show()

# Rating mediu per film
avg_ratings = spark.sql("""
    SELECT r.movieId, m.title, AVG(r.rating) AS avg_rating
    FROM ratings r
    JOIN movies m ON r.movieId = m.movieId
    GROUP BY r.movieId, m.title
    ORDER BY avg_rating DESC
""")
avg_ratings.show()


+-------+---------------+-------+
|movieId|          title|  genre|
+-------+---------------+-------+
|      1|The Dark Knight|Acțiune|
|      5|       Die Hard|Acțiune|
|      9|   Monty Python|Acțiune|
|     13|  The Godfather|Acțiune|
|     17|   Finding Nemo|Acțiune|
+-------+---------------+-------+

+-------+--------------------+------------------+
|movieId|               title|        avg_rating|
+-------+--------------------+------------------+
|      5|            Die Hard|               4.0|
|      6|            Superbad|3.7249999999999996|
|     10|     Dumb and Dumber|3.5999999999999996|
|     12|        Forrest Gump|               3.3|
|      3|          The Matrix|3.0799999999999996|
|     16|           Toy Story|2.8666666666666667|
|      1|     The Dark Knight|              2.85|
|     17|        Finding Nemo| 2.733333333333333|
|     11|The Shawshank Red...|2.6714285714285713|
|     14|    Schindler's List|2.6000000000000005|
|     18|       The Lion King| 2.2666666666